In [1]:
##### Calculates production totals for each sub-national region in the study
# calculates raster stats of production statistics for totals, crop v livestock, and individual commodities 

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
from rasterio.warp import reproject, Resampling
import rasterstats
from rasterstats import zonal_stats

In [2]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

# load sub-national geographies
total_geo = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_total.shp")

# Save paths
capital_path_target = f"{cd}/Data/Clean/Production/Subnational_stats/subnational_production_stats_capital_regions_target.csv"
labor_path_target = f"{cd}/Data/Clean/Production/Subnational_stats/subnational_production_stats_labor_regions_target.csv"

capital_path_predictor = f"{cd}/Data/Clean/Production/Subnational_stats/subnational_production_stats_capital_regions_predictor.csv"
labor_path_predictor = f"{cd}/Data/Clean/Production/Subnational_stats/subnational_production_stats_labor_regions_predictor.csv"

In [3]:
##### Combine crop and livestock production ($) into total 

livestock_path = f"{cd}/Data/Clean/Production/livestock_production_USD_2020.tif"
crop_path = f"{cd}/Data/Clean/Production/crop_production_USD_2020.tif"
output_path = f"{cd}/Data/Clean/Production/total_production_USD_2020.tif"

# resample crop raster to match livestock grid
with rasterio.open(livestock_path) as livestock_src:
    livestock = livestock_src.read(1).astype(np.float64)
    livestock[livestock == livestock_src.nodata] = np.nan
    meta = livestock_src.meta.copy()
    dst_crs = livestock_src.crs
    dst_transform = livestock_src.transform
    dst_shape = livestock_src.shape

with rasterio.open(crop_path) as crop_src:
    crops_aligned = np.full(dst_shape, np.nan, dtype=np.float64)
    reproject(
        source=rasterio.band(crop_src, 1),
        destination=crops_aligned,
        dst_crs=dst_crs,
        dst_transform=dst_transform,
        resampling=Resampling.sum  
    )
    crops_aligned[crops_aligned == crop_src.nodata] = np.nan

# combine rasters — NaN only if both are NaN or if livestock is NaN (because only livestock NaNs are in missing countries)
total = np.where(np.isnan(crops_aligned), livestock, livestock + crops_aligned)

meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")
out_arr = total.astype(np.float32)
out_arr[np.isnan(out_arr)] = -9999

with rasterio.open(output_path, "w", **meta) as dst:
    dst.write(out_arr, 1)

In [4]:
##### Combine crop and livestock production (t) into total 

livestock_path = f"{cd}/Data/Clean/Production/livestock_production_tonnes_2020.tif"
crop_path = f"{cd}/Data/Clean/Production/crop_production_tonnes_2020.tif"
output_path = f"{cd}/Data/Clean/Production/total_production_tonnes_2020.tif"

# resample crop raster to match livestock grid
with rasterio.open(livestock_path) as livestock_src:
    livestock = livestock_src.read(1).astype(np.float64)
    livestock[livestock == livestock_src.nodata] = np.nan
    meta = livestock_src.meta.copy()
    dst_crs = livestock_src.crs
    dst_transform = livestock_src.transform
    dst_shape = livestock_src.shape

with rasterio.open(crop_path) as crop_src:
    crops_aligned = np.full(dst_shape, np.nan, dtype=np.float64)
    reproject(
        source=rasterio.band(crop_src, 1),
        destination=crops_aligned,
        dst_crs=dst_crs,
        dst_transform=dst_transform,
        resampling=Resampling.sum  
    )
    crops_aligned[crops_aligned == crop_src.nodata] = np.nan

# combine rasters — NaN propagates naturally where either input is missing
total = np.where(np.isnan(crops_aligned), livestock, livestock + crops_aligned)

meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")
out_arr = total.astype(np.float32)
out_arr[np.isnan(out_arr)] = -9999

with rasterio.open(output_path, "w", **meta) as dst:
    dst.write(out_arr, 1)

In [5]:
##### Define dict of rasters to use for target variables

raster_dict_target = {
    # Aggregated production (value)
    "total_production_USD"     : f"{cd}/Data/Clean/Production/total_production_USD_2020.tif",
    "livestock_production_USD" : f"{cd}/Data/Clean/Production/livestock_production_USD_2020.tif",
    "crop_production_USD"      : f"{cd}/Data/Clean/Production/crop_production_USD_2020.tif",
    # Aggregated production (t)
    "total_production_t"     : f"{cd}/Data/Clean/Production/total_production_tonnes_2020.tif",
    "livestock_production_t" : f"{cd}/Data/Clean/Production/livestock_production_tonnes_2020.tif",
    "crop_production_t"      : f"{cd}/Data/Clean/Production/crop_production_tonnes_2020.tif",
}

In [6]:
##### Define function to get rasterstats 
# Calculates the sum of raster values within each polygon in a GeoDataFrame and adds the result as a new column

def subnational_production_stats(geo, raster_path, column_name, nodata=-9999):

    with rasterio.open(raster_path) as src:
        raster_crs = src.crs

    # ensure crs match
    gdf_proj = geo.to_crs(raster_crs)

    zonal = rasterstats.zonal_stats(
        gdf_proj,
        raster_path,
        stats="sum",
        nodata=nodata
    )

    geo[column_name] = [z["sum"] for z in zonal]
    return geo


In [8]:
### Run raster_stats for each target raster
for column_name, raster_path in raster_dict_target.items():
    total_geo_target = subnational_production_stats(total_geo, raster_path, column_name)

In [9]:
### Final clean and save target

col_to_keep_target = [
    # identifiers
    'PROJ_ID', 'subset',
    # aggregated production value
    'total_production_USD', 'livestock_production_USD', 'crop_production_USD',
    # aggregated production weight
    'total_production_t', 'livestock_production_t', 'crop_production_t',
]

total_geo_target_test = total_geo_target[col_to_keep_target]

# split into labor and capital
capital_geo_target = total_geo_target_test[total_geo_target_test['subset'].isin(['capital', 'both'])]
labor_geo_target = total_geo_target_test[total_geo_target_test['subset'].isin(['labor', 'both'])]

### Save
capital_geo_target.to_csv(capital_path_target, index=False)
labor_geo_target.to_csv(labor_path_target, index=False)